In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

path = kagglehub.competition_download('titanic')

print("Path to competition files:", path)

train = pd.read_csv('/kaggle/input/competitions/titanic/train.csv')
test = pd.read_csv('/kaggle/input/competitions/titanic/test.csv')

test.head()
train.head()

/kaggle/input/competitions/titanic/train.csv
/kaggle/input/competitions/titanic/test.csv
/kaggle/input/competitions/titanic/gender_submission.csv
Path to competition files: /kaggle/input/competitions/titanic


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [2]:
# percentage of women who has survived
total_women = train.loc[train.Sex=='female']
Survived_women = total_women.loc[(total_women.Survived == 1)]

Percentage = len(Survived_women)/len(total_women)
print(f"Women Perentage : {Percentage:.2%}")

# percentage of men who has survived
total_men = train.loc[train.Sex=='male']
Survived_men = total_men.loc[(total_men.Survived == 1)]

Percentage = len(Survived_men)/len(total_men)
print(f"men Perentage : {Percentage:.2%}")

Women Perentage : 74.20%
men Perentage : 18.89%


In [3]:
# Feature Engineering

# Create a list containing both datasets
dfs = [train, test]

for df in dfs:
    # 1. Extract Title
    df["Title"] = df["Name"].str.extract(r" ([A-Za-z]+)\.", expand=False)
    df["Title"] = df["Title"].replace(
        [
            "Lady",
            "Countess",
            "Capt",
            "Col",
            "Don",
            "Dr",
            "Major",
            "Rev",
            "Sir",
            "Jonkheer",
            "Dona",
        ],
        "Rare",
    )
    df["Title"] = df["Title"].replace(
        {"Mlle": "Miss", "Ms": "Miss", "Mme": "Mrs"}
    )

    # 2. Family Features
    df["FamilySize"] = df["SibSp"] + df["Parch"] + 1
    df["IsAlone"] = (df["FamilySize"] == 1).astype(int)

    df["Age"] = df.groupby(["Pclass", "Sex"])["Age"].transform(
        lambda x: x.fillna(x.median())
    )

    


In [4]:
from sklearn.ensemble import RandomForestClassifier

y = train["Survived"]

features = ["Pclass", "Sex","Embarked","SibSp",
    "Parch","Title","FamilySize","IsAlone", "Age","Fare"]
X = pd.get_dummies(train[features])
X_test = pd.get_dummies(test[features])

model = RandomForestClassifier(n_estimators=200, max_depth=4, random_state=1)



In [5]:
# Run K-Fold Cross-Validation
from sklearn.model_selection import cross_validate

cv_results = cross_validate(
    model, X, y, cv=5, scoring="accuracy", return_train_score=True
)

# Calculate averages
train_score = cv_results["train_score"].mean()
cv_score = cv_results["test_score"].mean()

print(f"Train Accuracy: {train_score:.4f}")
print(f"CV Accuracy:    {cv_score:.4f}")
print(f"Gap (Overfit):  {train_score - cv_score:.4f}")


Train Accuracy: 0.8367
CV Accuracy:    0.8350
Gap (Overfit):  0.0017


In [6]:
model.fit(X, y)
predictions = model.predict(X_test)

output = pd.DataFrame({'PassengerId': test.PassengerId, 'Survived': predictions})
output.to_csv('submission.csv', index=False)
print("Your submission was successfully saved!")

Your submission was successfully saved!
